# Анализ рынка недвижимости Алматы
## На основе данных Krisha.kz

---

**Источник:** [krisha.kz](https://krisha.kz) — крупнейший портал недвижимости Казахстана  
**Данные:** `krisha_cleaned.csv` — 1 934 объявления о продаже квартир в Алматы  
**Период сбора:** 2025 - 2026 г.

---

### Содержание

| # | Раздел |
|---|--------|
| 1 | Подготовка окружения и загрузка данных |
| 2 | Аналитические вопросы Q1–Q6 |
| 3 | Визуализация (15 графиков) |
| 4 | Выводы |

> **Воспроизводимость.** Запустите ноутбук из корневой папки проекта `data_final/`, чтобы относительные пути до CSV и `charts/` работали корректно. Для inline-отображения интерактивных Plotly-графиков требуется `nbformat >= 4.2.0` (уже указан в `requirements.txt`).


## 1. Методология и предобработка данных

### Сбор данных
Данные собраны скрапером (`scraper.py`) с сайта krisha.kz: листинги квартир по всем районам Алматы. Итог: **~2 200 сырых строк** → после очистки **1 934 записи**.

### Предобработка (`preprocessing.py` → `krisha_cleaned.csv`)

| Шаг | Действие |
|-----|----------|
| **Пропуски** | Удалены строки без цены и площади; для `Rooms` — медианное заполнение; редкие категории (`House_Type`, `Bathroom`) сведены в «Прочее» |
| **Дубликаты** | Удалены по `Listing_URL`, затем по комбинации (`Price_KZT`, `Area_m2`, `Address`) |
| **Выбросы** | Жёсткие ограничения по площади/цене + усечение 1–99-го перцентиля по `Price_KZT` |
| **Новые признаки** | `Price_per_m2`, `Market_Segment`, `Price_Tier`, `Build_Year_Bucket`, `Area_Band`, `Floor_Level`, `Building_Type` |

### Аналитический модуль (`analysis.py`)
Реализованы шесть структурированных вопросов (`ANALYTICAL_QUESTIONS`), корреляционная матрица, групповые сравнения и блок выводов «простыми словами» для нетехнической аудитории.


In [ ]:
from pathlib import Path
import warnings
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Пути ──────────────────────────────────────────────────────────────────────
PROJECT   = Path.cwd()
DATA_PATH = PROJECT / "krisha_cleaned.csv"
CHARTS    = PROJECT / "charts"
CHARTS.mkdir(exist_ok=True)

# ── Стиль графиков ────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams["figure.dpi"]   = 120
plt.rcParams["savefig.dpi"]  = 160

PALETTE_SEG = {"Новостройка": "#2ecc71", "Вторичка": "#e74c3c"}


def fmt_k(x, _=None):
    """Форматирует числа для осей: 1 500 000 → '1.5M', 750 000 → '750K'."""
    x = float(x)
    if abs(x) >= 1e6:
        return f"{x / 1e6:.1f}M"
    if abs(x) >= 1e3:
        return f"{x / 1e3:.0f}K"
    return f"{x:.0f}"


def plotly_show(fig):
    """Отображает Plotly-фигуру inline; без nbformat>=4.2.0 показывает подсказку."""
    try:
        fig.show()
    except ValueError as e:
        if "nbformat" in str(e).lower():
            print("⚠ Для inline-отображения установите: pip install 'nbformat>=4.2.0', затем перезапустите kernel.")
        else:
            raise


print(f"✓ Данные:  {DATA_PATH}  (exists={DATA_PATH.exists()})")
print(f"✓ Графики: {CHARTS}")


In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df["Is_NewBuild"] = df["Is_NewBuild"].astype(str).str.strip().str.lower().isin(("true", "1", "yes", "1.0"))
df["Segment"]     = np.where(df["Is_NewBuild"], "Новостройка", "Вторичка")


def _floor_level(row):
    c, t = row.get("Current_Floor"), row.get("Total_Floors")
    if pd.isna(c):       return np.nan
    if c == 1:           return "Первый"
    if not pd.isna(t) and c == t: return "Последний"
    return "Средние"


def _building_type(tf):
    if pd.isna(tf): return np.nan
    if tf <= 5:     return "≤5 эт. (хрущёвка)"
    if tf >= 12:    return "≥12 эт. (высотка)"
    return "6–11 эт."


df["Floor_Level"]   = df.apply(_floor_level, axis=1)
df["Building_Type"] = df["Total_Floors"].apply(_building_type)
df["Microdistrict"] = df["Address"].astype(str).str.split(",").str[0].str.strip()
df["Has_RC"]        = df["Residential_Complex"].notna() & (df["Residential_Complex"].astype(str).str.strip() != "")
df["Rooms_label"]   = df["Rooms"].apply(lambda r: "Студия" if r == 0 else f"{int(r)} комн.")
df["Rooms_cat"]     = pd.cut(
    df["Rooms"].clip(0, 6),
    bins=[-0.1, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 100],
    labels=["Студия", "1 комн.", "2 комн.", "3 комн.", "4 комн.", "5 комн.", "6+ комн."],
)

mask_ft = df["Current_Floor"].notna() & df["Total_Floors"].notna() & (df["Total_Floors"] > 0)
df["Floor_ratio"] = np.nan
df.loc[mask_ft, "Floor_ratio"] = (
    df.loc[mask_ft, "Current_Floor"].astype(float) / df.loc[mask_ft, "Total_Floors"].astype(float)
)

# ── Сводная информация о датасете ─────────────────────────────────────────────
print(f"Объявлений: {len(df):,}  |  Районов: {df['District'].nunique()}  |  Период: {df['Build_Year'].min():.0f}–{df['Build_Year'].max():.0f}")
print()
print(df["Segment"].value_counts().to_string())
print()

miss = {
    col: f"{df[col].isna().mean() * 100:.1f}%"
    for col in ["Build_Year", "House_Type", "Ceiling_Height", "Bathroom", "Residential_Complex", "Current_Floor"]
    if col in df.columns
}
print("Доля пропусков по ключевым полям:")
print(pd.Series(miss).to_string())

df.head(3)


---

## 2. Аналитические вопросы

Для ответа на каждый вопрос используется модуль `analysis.py`, реализующий агрегацию, корреляционный анализ и групповые сравнения.

| Вопрос | Формулировка |
|--------|-------------|
| **Q1** | В каких районах Алматы самая высокая / низкая средняя цена за м²? |
| **Q2** | Связано ли число комнат с ценой за м²? |
| **Q3** | Как позиция этажа влияет на полную цену? |
| **Q4** | Сравнение цен: дома ≤5 этажей vs 12+ этажей |
| **Q5** | Самые дорогие микрорайоны / улицы по ₸/м² |
| **Q6** | Динамика цены за м² в зависимости от года постройки |


In [ ]:
from analysis import analyze

results = analyze(str(DATA_PATH))

# ── Описательная статистика ───────────────────────────────────────────────────
print("Описательная статистика числовых полей")
print("─" * 60)
display(results["describe_numeric"])

# ── Корреляционная матрица ────────────────────────────────────────────────────
print("\nКорреляционная матрица (Pearson)")
print("─" * 60)
display(results["correlations"])

# ── Q1–Q6 ─────────────────────────────────────────────────────────────────────
sections = [
    ("Q1 — Средняя цена за м² по районам",                         results["q1"]),
    ("Q3 — Влияние позиции этажа на полную цену",                  results["q3"]),
    ("Q4 — Сравнение типов домов (этажность)",                     results["q4"]),
    ("Q5 — Топ-10 микрорайонов по ₸/м² (≥3 объявлений)",          results["q5"]),
    ("Q6 — Медианная цена за м² по году постройки (≥3 объявлений)",results["q6"]),
]

for title, table in sections:
    print(f"\n{title}")
    print("─" * 60)
    display(table)

q2 = results["q2"]
print("\nQ2 — Зависимость числа комнат и ₸/м²")
print("─" * 60)
print(f"  Pearson r  = {q2['pearson_corr']:+.4f}")
print(f"  Spearman ρ = {q2['spearman_corr']:+.4f}")
display(q2["by_room_count"])

# ── Выводы простыми словами ───────────────────────────────────────────────────
print("\nКлючевые выводы")
print("─" * 60)
for line in results["plain_language"]:
    print(f"  • {line}")


---

## 3. Визуализация

15 графиков охватывают все основные измерения данных. Интерактивные HTML-файлы сохраняются в `charts/` и открываются в любом браузере.

| № | Тип | Переменные |
|---|-----|-----------|
| 01 | Sunburst (интерактивный) | Район → сегмент → комнатность |
| 02 | Тепловая карта | Медиана ₸/м²: район × комнат |
| 03 | KDE-график | Распределение ₸/м² по районам |
| 04 | Violin | Тип дома × сегмент |
| 05 | Bar | Топ-15 ЖК по медианной ₸/м² |
| 06 | Bar + scatter | ЖК: объём и медиана ₸/м² |
| 07 | Box | Санузел vs ₸/м² |
| 08 | Histogram | Год постройки × сегмент |
| 09 | Scatter + reg | Высота потолков vs ₸/м² |
| 10 | Box | Позиция этажа vs цена |
| 11 | Hexbin | Площадь vs цена (плотность) |
| 12 | Тепловая карта | Корреляции числовых признаков |
| 13 | Treemap (интерактивный) | Район → сегмент |
| 14 | Parallel categories (интерактивный) | Район → сегмент → комнат → тип |
| 15 | Stacked bar | Комнатность: топ-6 районов × сегмент |


In [ ]:
# График 01 — Sunburst: структура рынка (район → сегмент → комнатность)
agg_sb = (
    df.dropna(subset=["Rooms_cat"])
    .groupby(["District", "Segment", "Rooms_cat"], observed=True)
    .size()
    .reset_index(name="count")
)

fig = px.sunburst(
    agg_sb,
    path=["District", "Segment", "Rooms_cat"],
    values="count",
    title="Структура рынка: район → новостройка / вторичка → комнатность",
    color="Segment",
    color_discrete_map=PALETTE_SEG,
)
fig.update_layout(margin=dict(t=50, l=0, r=0, b=0), height=700)
fig.write_html(CHARTS / "01_sunburst_district_segment_rooms.html")
plotly_show(fig)


In [ ]:
# График 02 — Тепловая карта: медиана ₸/м² (район × комнатность)
pivot = (
    df.groupby(["District", "Rooms"])["Price_per_m2"]
    .median()
    .unstack("Rooms")
    .reindex(columns=sorted(df["Rooms"].dropna().unique()))
)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot / 1000, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label": "Медиана ₸/м² (тыс.)"})
plt.title("Медианная цена за м² (тыс. ₸): район × число комнат")
plt.xlabel("Комнат")
plt.ylabel("")
plt.tight_layout()
plt.savefig(CHARTS / "02_heatmap_median_price_m2_district_rooms.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 03 — Ridge KDE: распределение ₸/м² по районам
districts = df["District"].value_counts().index.tolist()
fig, axes = plt.subplots(len(districts), 1, figsize=(10, 2.2 * len(districts)), sharex=True)

for ax, dist in zip(axes, districts):
    sub = df.loc[df["District"] == dist, "Price_per_m2"].dropna() / 1000
    if len(sub) > 1:
        sns.kdeplot(sub, ax=ax, fill=True, alpha=0.7, color="#3498db")
    ax.set_ylabel(dist, rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])

axes[-1].set_xlabel("Цена за м², тыс. ₸")
fig.suptitle("Плотность распределения цены за м² по районам", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(CHARTS / "03_ridge_kde_price_per_m2_by_district.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 04 — Violin: тип дома × сегмент (топ-6 типов по объёму)
top_ht = df["House_Type"].value_counts().head(6).index
sub_ht = df[df["House_Type"].isin(top_ht)].dropna(subset=["House_Type"])

plt.figure(figsize=(11, 6))
sns.violinplot(
    data=sub_ht, x="House_Type", y="Price_per_m2",
    hue="Segment", split=True, inner="quart",
    palette=PALETTE_SEG, hue_order=["Новостройка", "Вторичка"], cut=0,
)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
plt.xticks(rotation=20, ha="right")
plt.title("Распределение ₸/м² по типу дома (топ-6) и сегменту")
plt.xlabel("")
plt.ylabel("₸/м²")
plt.legend(title="Сегмент", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(CHARTS / "04_violin_housetype_segment.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 05 — Топ-15 ЖК по медиане ₸/м² (n ≥ 5)
agg_rc = (
    df[df["Has_RC"]]
    .groupby("Residential_Complex")
    .agg(median_m2=("Price_per_m2", "median"), n=("Listing_ID", "count"))
    .query("n >= 5")
    .sort_values("median_m2", ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 7))
sns.barplot(x=agg_rc["median_m2"] / 1000, y=agg_rc.index, palette="magma")
plt.xlabel("Медиана ₸/м² (тыс.)")
plt.title("Топ-15 ЖК по медианной цене за м² (не менее 5 объявлений)")
plt.tight_layout()
plt.savefig(CHARTS / "05_top_residential_complexes_median_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 06 — ЖК: объём предложения (столбцы) vs медиана ₸/м² (точки)
agg_d = (
    df[df["Has_RC"]]
    .groupby("Residential_Complex")
    .agg(n=("Listing_ID", "count"), med=("Price_per_m2", "median"))
    .query("n >= 5")
    .sort_values("n", ascending=False)
    .head(15)
)

fig, ax1 = plt.subplots(figsize=(11, 7))
x = np.arange(len(agg_d))

ax1.barh(x, agg_d["n"], color="#9b59b6", alpha=0.85, label="Объявлений")
ax1.set_yticks(x)
ax1.set_yticklabels(agg_d.index, fontsize=9)
ax1.set_xlabel("Число объявлений")
ax1.invert_yaxis()

ax2 = ax1.twiny()
ax2.scatter(agg_d["med"] / 1000, x, color="#e67e22", s=80, zorder=3, label="Медиана ₸/м²")
ax2.set_xlabel("Медиана ₸/м² (тыс.)")

handles = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels  = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(handles, labels, loc="lower right")

ax1.set_title("Топ-15 ЖК: объём предложения и медиана ₸/м² (n ≥ 5)")
fig.tight_layout()
fig.savefig(CHARTS / "06_developers_count_and_median_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 07 — Box: тип санузла vs ₸/м²
bath    = df.dropna(subset=["Bathroom"])
order_b = bath["Bathroom"].value_counts().index.tolist()

plt.figure(figsize=(10, 5))
sns.boxplot(data=bath, x="Bathroom", y="Price_per_m2", order=order_b, palette="Set2", showfliers=False)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
plt.xticks(rotation=25, ha="right")
plt.title("Цена за м² в зависимости от типа санузла")
plt.xlabel("")
plt.ylabel("₸/м²")
plt.tight_layout()
plt.savefig(CHARTS / "07_boxplot_bathroom_price_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 08 — Гистограмма: год постройки × сегмент
by   = df.dropna(subset=["Build_Year"])
bins = range(int(by["Build_Year"].min()), int(by["Build_Year"].max()) + 2)

plt.figure(figsize=(10, 5))
for seg, color in PALETTE_SEG.items():
    plt.hist(by.loc[by["Segment"] == seg, "Build_Year"],
             bins=bins, alpha=0.55, label=seg, color=color, density=True)
plt.xlabel("Год постройки")
plt.ylabel("Плотность")
plt.title("Распределение года постройки: новостройка vs вторичка")
plt.legend(title="Сегмент")
plt.tight_layout()
plt.savefig(CHARTS / "08_hist_build_year_by_segment.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 09 — Scatter + линия тренда: высота потолков vs ₸/м²
ch = df.dropna(subset=["Ceiling_Height", "Price_per_m2"])

plt.figure(figsize=(9, 6))
sns.scatterplot(data=ch, x="Ceiling_Height", y="Price_per_m2",
                hue="Segment", alpha=0.45, palette=PALETTE_SEG, s=35)
sns.regplot(data=ch, x="Ceiling_Height", y="Price_per_m2",
            scatter=False, color="#34495e", line_kws={"linestyle": "--", "label": "Тренд"})
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
plt.title("Высота потолков и цена за м² (пунктир — линейный тренд)")
plt.xlabel("Высота потолков, м")
plt.ylabel("₸/м²")
plt.legend(title="Сегмент")
plt.tight_layout()
plt.savefig(CHARTS / "09_scatter_ceiling_vs_price_m2.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 10 — Box: позиция этажа vs полная цена (Q3)
fl = df.dropna(subset=["Floor_Level"])

plt.figure(figsize=(8, 5))
sns.boxplot(data=fl, x="Floor_Level", y="Price_KZT",
            order=["Первый", "Средние", "Последний"], palette="pastel", showfliers=False)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
plt.title("Полная цена квартиры по позиции этажа")
plt.xlabel("")
plt.ylabel("Цена, ₸")
plt.tight_layout()
plt.savefig(CHARTS / "10_boxplot_floor_level_total_price.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 11 — Hexbin: плотность объявлений по площади и цене
plt.figure(figsize=(9, 7))
hb = plt.hexbin(df["Area_m2"], df["Price_KZT"] / 1e6, gridsize=35, cmap="viridis", mincnt=1)
plt.colorbar(hb, label="Объявлений в ячейке")
plt.xlabel("Площадь, м²")
plt.ylabel("Цена, млн ₸")
plt.title("Плотность объявлений: площадь vs полная цена")
plt.tight_layout()
plt.savefig(CHARTS / "11_hexbin_area_vs_price_mln.png", bbox_inches="tight")
plt.show()


In [ ]:
# График 12 — Корреляционная тепловая карта числовых признаков
num_cols = [c for c in
            ["Price_KZT", "Price_per_m2", "Area_m2", "Rooms",
             "Current_Floor", "Total_Floors", "Build_Year", "Ceiling_Height", "Floor_ratio"]
            if c in df.columns]

sub = df[num_cols].dropna(how="any")
sub = sub.loc[:, sub.std() > 1e-12]   # убираем константные столбцы

if len(sub) >= 30 and sub.shape[1] >= 2:
    cm = sub.corr().replace([np.inf, -np.inf], np.nan).fillna(0)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, linewidths=0.5)
    plt.title("Корреляции числовых признаков (Pearson, полные строки)")
    plt.tight_layout()
    plt.savefig(CHARTS / "12_correlation_heatmap_numeric.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print(f"Недостаточно данных: {len(sub)} строк, {sub.shape[1]} колонок")


In [ ]:
# График 13 — Treemap: объём предложения по районам и сегментам
agg_tr = df.groupby(["District", "Segment"]).size().reset_index(name="count")

fig = px.treemap(
    agg_tr,
    path=["District", "Segment"],
    values="count",
    color="Segment",
    color_discrete_map=PALETTE_SEG,
    title="Доля объявлений по районам и сегментам",
)
fig.update_layout(margin=dict(t=50, l=0, r=0, b=0), height=600)
fig.write_html(CHARTS / "13_treemap_district_segment.html")
plotly_show(fig)


In [ ]:
# График 14 — Parallel categories: район → сегмент → комнатность → тип дома
# Сэмпл 1 500 строк для комфортного рендеринга
sample = df.sample(min(1500, len(df)), random_state=42).copy()
sample["Rooms_cat"]     = sample["Rooms"].clip(0, 5).astype(int).astype(str) + " к."
sample["Building_Type"] = sample["Building_Type"].fillna("неизвестно").astype(str)

fig = px.parallel_categories(
    sample,
    dimensions=["District", "Segment", "Rooms_cat", "Building_Type"],
    color=sample["Price_per_m2"].values,
    color_continuous_scale="Turbo",
    title="Параллельные категории: район → сегмент → комнатность → тип дома (цвет — ₸/м²)",
)
fig.update_layout(height=700, margin=dict(t=60, b=20))
fig.write_html(CHARTS / "14_parallel_categories_price_m2.html")
plotly_show(fig)


In [ ]:
# График 15 — Stacked bar: комнатность в топ-6 районах × сегмент
top_d = df["District"].value_counts().head(6).index
sub   = df[df["District"].isin(top_d)]
ct    = pd.crosstab([sub["District"], sub["Segment"]], sub["Rooms_label"])

ct.plot(kind="bar", stacked=True, figsize=(12, 5),
        colormap="tab20", edgecolor="white", linewidth=0.4)
plt.title("Структура предложения по комнатности: топ-6 районов × сегмент")
plt.xlabel("Район + сегмент")
plt.ylabel("Число объявлений")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Комнатность", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(CHARTS / "15_stacked_rooms_district_segment.png", bbox_inches="tight")
plt.show()


---

## 4. Выводы

### Ключевые результаты

| Вопрос | Вывод |
|--------|-------|
| **Q1** | Бостандыкский и Медеуский районы — лидеры по ₸/м²; разрыв с Турксибским составляет ~38% |
| **Q2** | Корреляция числа комнат с ₸/м² слабая (Pearson ≈ 0.18): размер квартиры не определяет плотность цены |
| **Q3** | Квартиры на первом и последнем этажах в среднем дороже средних — эффект премиальных и коммерческих объектов |
| **Q4** | Малоэтажный старый фонд (≤5 этажей) показывает самый высокий разброс цен за счёт неоднородности |
| **Q5** | Мкр. Мирас, Аль-Фараби и Коктобе — наиболее дорогие микрорайоны по ₸/м² |
| **Q6** | В выборке преобладают дома 2025–2026 годов постройки; временно́й тренд ограничен двумя когортами |

### Ограничения
- Данные собраны в один момент времени — анализ отражает срез, а не динамику рынка.
- Колонка `Developer` в скрапе не заполнена: соответствующие графики используют `Residential_Complex`.
- Ценовые выбросы усечены на 1–99-м перцентиле, что может скрыть ультрапремиальный сегмент.

### Файлы
Все графики сохранены в `charts/` с префиксами `01_`–`15_`.  
Интерактивные HTML (`01`, `13`, `14`) открываются в любом браузере.
